# 임베딩 모델 파인튜닝

- 임베딩 모델 파인튜닝은 사전 학습된 임베딩 모델을 특정 도메인이나 작업에 맞게 최적화하는 과정입니다.


## 1. 임베딩 모델의 학습 원리

- 의미가 비슷한 문장 쌍에는 높은 임베딩 유사도를, 의미가 다른 문장 쌍에는 낮은 유사도를 반환하도록 임베딩 벡터를 업데이트 하는 방식입니다.
- 임베딩 모델을 학습할 때는 의미가 유사한 문장 쌍과 유사하지 않은 문장 쌍을 대조하여 학습하는 방식, 즉 대조 학습을 활용합니다.


### 1.1 대조학습

- 포지티브 샘플
  - 의미적으로 관련이 있는 문장 쌍을 의미합니다.
  - 예: (기준 문서: "서울의 인구는?", 비교 문서: "서울의 인구는 약 970만명 입니다.")
- 네커티브 샘플
  - 기준 문서는 동일하지만, 비교 문서는 의미적으로 관련이 없거나 관련성이 낮은 문장을 준비하여 이들을 쌍으로 구성한 데이터입니다.
  - 예: (기준 문서: "서울의 인구는?", 비교 문서: "파리는 프랑스의 수도입니다.")


- 포지티브 샘플은 RAG를 수행할 때 사용자가 입력할 만한 검색어를 기준문서, 검색 결과로 유사도가 높게 나오기를 바라는 문서를 관련 있는 문서로 삼아 구성합니다.
- 네거티브 샘플은 RAG상황에서 같은 앵커에 대한 검색 결과에 포함되지 않기를 바라는 문서를 짝지어 구성합니다.


- 포지티브 샘플과 네거티브 샘플 구성
  - 기준 문서를 중심으로 유사도가 높은 쌍인 포지티브 샘플(관련 있는 쌍)과 유사도가 낮은 네거티브 샘플(관련 없는 쌍)을 모두 학습 데이터로 준비합니다.
- 대조 학습
  - 모델이 포지티브 샘플 쌍의 임베딩 간 거리는 가깝게, 네거티브 샘플 쌍의 임베딩 간 거리는 멀게 만들도록 학습합니다.
- 손실 함수 최적화
  - 임베딩 간 유사도를 계산하여, 포지티브 쌍의 임베딩 유사도는 높이고 네거티브 쌍의 임베딩 유사도는 낮추는 방향으로 손실 함수를 최적화합니다.


- 손실 함수는 모델이 예측한 결과와 실제 정답 간의 오차를 계산해 학습을 조정하는 기준이 됩니다.
- MultipleNegativesRankingLoss라는 손실 함수를 사용할 예정입니다.


### 1.2 데이터셋 구성

- 대조 학습에서는 하나의 기준 문서에 대해 하나의 포지티브 샘플과 하나 이상의 네거티브 샘플을 명시적으로 준비해야 합니다.
- 기준 문서는 앵커라고 부릅니다.
- 네거티브 샘플은 포지티브 샘플보다 양이 많을수록 좋습니다.


- 트리플렛 구성
  - 전통적인 방식은 각 학습 데이터를 (앵커, 포지티브, 네거티브) 형태의 트리플렛으로 구성하는 것입니다.


In [ ]:
# 전통적인 트리플렛 구성 예
triplets = [
    # (앵커, 포지티브, 네거티브)
    ("강아지를 기르는 방법", "반려견 양육 가이드", "고양이 사료 추천"),
    ("파이썬 코딩 튜토리얼", "파이썬 프로그래밍 기초", "자바스크립트 입문 강의"),
    # 수천, 수만 개의 트래플렛 필요
]

- 다중 네거티브 구성
  - 실제 모델 학습에서는 하나의 앵커에 여러개의 네거티브 샘플을 포함하는 구성이 더 효과적인 경우가 많습니다.


In [ ]:
# 다중 네거티브 샘플 구성 예
training_data = [
    {
        "anchor": "머신러닝이란?",
        "positive": "기계학습은 데이터로부터 패턴을 찾는 AI 기술입니다.",
        "negatives": [
            "오늘 날씨가 좋네요",
            "내일 회의는 2시에 시작합니다.이 식당의 불고기가 맛있습니다.",
            # 여러 개의 네거티브 샘플
        ],
    },
    # 수천개의 이러한 구조
]

- 다중 네거티브 구성은 학습 효과를 높일 수 있지만, 그만큼 데이터 준비의 난이도도 높아집니다.
- 임베딩 모델을 효과적으로 파인튜닝하기 위해서는, 특히 네거티브 샘플 선정이 가장 까다로운 작업중 하나입니다.


- 네거티브 샘플의 문서는 각 앵커와 관련 없는 텍스트여야만 합니다.
- 적절한 난이도의 네거티브 샘플을 선택해야 합니다.
  - 두 개의 쌍이 너무 관련이 없다면 임베딩 모델이 판단하기 너무 쉬워서 학습 효과가 거의 없게 됩니다.
  - 사람이 보아도 관련이 있는 것인지 관련이 없는 것인지 헷갈릴 정도의 문서 쌍이라면 난이도가 너무 높아져 학습에 오히려 방해가 됩니다.
- 다중 네거티브 샘플을 구성할 경우, 네거티브 샘플을 포지티브 샘플 대비 몇 배로 구성하느냐에 따라 데이터셋 크기가 기하급수적으로 증가하고, 만들어야 하는 데이터의 양이 많아지게 됩니다.


- 임베딩 파인튜닝에서 양질의 네거티브 샘플을 구성하는 것은 종종 전체 학습과정에서 가장 어려운 부분 중 하나입니다.


### 1.3 배치 내 네거티브 샘플링

- 배치 내에서 네거티브 샘플을 선정하는 학습 방법을 사용합니다.
- 이 방법은 명시적인 네거티브 샘플을 별도로 준비할 필요가 없다는 큰 장점이 있습니다.
- 학습 데이터에서 다른 앵커에서 사용하고 있는 샘플을 참고하여 자동으로 네거티브로 활용합니다.
- 이 원리를 이해하려면 배치라는 개념을 알아야 합니다.
- AI모델은 데이터를 적당한 개수의 묶음으로 나누어 학습합니다.
- 예를 들어 데이터가 5000개 이고 배치 크기를 40으로 설정했다면, 데이터를 40개씩 묶어 125회에 걸쳐 학습하게 됩니다.
- 배치란 모델이 한 번에 학습하는 데이터의 단위를 뜻하며, 병렬적으로 데이터를 몇개씩 학습할 것이냐를 의미합니다.


- 배치 내 네거티브 생성 방법은 포지티브 샘플만으로 데이터를 구성하더라도 배치내에서 네거티브 샘플들을 자동으로 만드는 학습 방법입니다.
- 사용자는 학습을 위해 포지티브 샘플만 제공하면 되며, 네거티브 샘플은 학습 시 배치내에서 자동으로 생성됩니다.


- 예를 들어 배치 크기가 4인 경우, 한번의 학습에 네 개의 서로 다른 앵커 문서와 그에 대응하는 포지티브 샘플이 사용되며, 이들 간 교차로 네거티브 샘플 역할도 동시에 수행됩니다.


```
배치 = [
    (앵커문서1, 문서1),
    (앵커문서2, 문서2),
    (앵커문서3, 문서3),
    (앵커문서4, 문서4)
]
```


- 앵커 문서1의 포지티브는 문서1, 나머지는 네거티브로 간주됩니다.
- 앵커 문서2의 포지티브는 문서2, 나머지는 네거티브로 간주됩니다.


- 하나의 배치 안에서 다른 쌍의 문서를 네거티브로 자동 활용하면서 대조 학습을 수행합니다.


```
[
    ("AI란 무엇인가?", "AI는 인간의 지능을 모방한 기술입니다."),
    ("딥러닝이란?", "신경망을 여러 층 쌓아 데이터로부터 학습하는 기계학습 방법입니다."),
    ("Python은 어디에 쓰이나요?", "Python은 데이터 분석, 웹 개발, AI 등에 널리 사용됩니다."),
    ("자연어 처리란?" , "컴퓨터가 인간의 언어를 이해하고 처리하는 AI의 한 분야입니다.")
]
```


### MultipleNegativesRankingLoss

- 이 손실 함수는 포지티브 샘플과의 유사도는 높이고, 네거티브 샘플과는 유사도는 낮추도록 설계되어 있습니다.


In [1]:
from sentence_transformers import SentenceTransformer, losses, InputExample
from torch.utils.data import DataLoader
import torch

# 모델 로드
model = SentenceTransformer("BAAI/bge-m3")

# 훈련 데이터 준비
train_examples = [
    InputExample(texts=["AI란 무엇인가?", "AI는 인간의 지능을 모방한 기술입니다."]),
    InputExample(
        texts=[
            "딥러닝이란?",
            "신경망을 여러 층 쌓아 데이터로부터 학습하는 기계학습 방법입니다.",
        ]
    ),
    InputExample(
        texts=[
            "Python은 어디에 쓰이나요?",
            "Python은 데이터 분석, 웹 개발, AI등에 널리 사용됩니다.",
        ]
    ),
    InputExample(
        texts=[
            "자연어 처리란?",
            "컴퓨터가 인간의 언어를 이해하고 처리하는 AI의 한 분야입니다.",
        ]
    ),
]

# 배치 크기가 클수록 성능이 향상될 수 있지만 GPU에 따라서 최대 비치 크기가 제한됨
batch_size = 32
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)

# MultipleNegativesRankingLoss 설정
# 온도(temperature) 파라미터를 조정하여 손실 함수의 강도 조절 가능
loss = losses.MultipleNegativesRankingLoss(
    model, scale=20.0
)  # scale은 temperature의 역수

# 학습 설정
train_loss = losses.MultipleNegativesRankingLoss(model)
warmup_steps = int(len(train_dataloader) * 0.1)  # 전체 훈련 데이터의 10%

# 모델 학습
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": 2e-5},
    output_path="./korean-sentence-embedding-model",
)

c:\workspace\python\rag_master\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\workspace\python\rag_master\.venv\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


- 먼저 sentence_transformers 라이브러리를 통해 BAAI/bge-m3 모델을 기본 모델로 로드합니다.
- 이어 훈련데이터를 InputExample객체의 리스트로 준비합니다.
- 모델은 이러한 문장의 쌍들을 통해 유사한 문장들이 임베딩 공간에서 가깝게 위치하도록 학습합니다.
- DataLoader를 활용하여 배치크기 32로 데이터를 효율적으로 처리하도록 설정합니다.
  - 데이터가 4개밖에 없지만, 실제 상황에서는 데이터가 32개보다 많다고 가정합니다.
- 손실함수로 MultipleNegativesRankingLoss를 채택했습니다. 의미적으로 유사한 문장들은 가깝게, 그렇지 않은 문장들은 멀리 위치시키도록 모델을 유도합니다.
  - scale=20.0 파라미터는 온도의 역수로 손실함숨의 강도를 적절히 조절하는 역할을 합니다.
- 학습과정에서는 워밍업단계를 전체 훈련데이터의 10%로 설정합니다.
- 업데이트 하는 정도를 조절하는 학습률은 2e-5 로 지정합니다.
- model.fit() 함수를 호출하여 모델을 학습합니다.
- 학습횟수를 의미하는 에포크의 경우 3
- 완성된 모델은 korean-sentence-embedding-model 디렉터리에 저장합니다.


## 2 학습 시 성능을 높이는 방법

### 3.1 배치 크기 키우기

- 임베딩 모델을 효과적으로 학습시키려면 배치 크기를 크게 설정하는 것이 중요한 전략 중 하나입니다.
- 대조 학습은 동일 앵커 기준으로 네거티브 샘플이 포지티브 샘플보다 많을수록 학습 성능이 올라간다는 특징이 있습니다.
- 배치 크기가 4인 경우: 각 질문에 대해 3개의 네거티브 샘플
- 배치 크기가 32인 경우: 각 질문에 대해 31개의 네거티브 샘플
- 배치 크기는 GPU 메모리 용량에 따라 제한되므로 무한정 키울수는 없습니다.
- 구글 코랩에서 제공되는 무료 GPU를 사용할 경우 배치 크기는 3~4수준에 그치는 경우가 많습니다.


### 2.2 하드 네거티브 선정

- 더 어려운 네거티브 샘플, 하드 네거티브를 추가하면 성능을 더욱 향상시킬 수 있습니다.
- 하드네거티브는 명시적으로 사용자가 직접 선택하여 학습 데이터에 포함시키는 네거티브 샘플을 의미합니다.


In [ ]:
# 하드 네거티브 예제
train_examples = [
    # (앵커, 포지티브, 하드 네거티브 형태로 제공)
    InputExample(
        texts=[
            "AI란 무엇인가?",
            "AI는 인간의 지능을 모방한 기술입니다.",
            "AI는 로봇과 같은 물리적 형태를 가진 기계입니다.",
        ]
    ),
    InputExample(
        texts=[
            "딥러닝이란?",
            "신경망을 여러 층 쌓아 데이터로부터 학습하는 기계학습 방법입니다.",
            "컴퓨터가 스스로 생각하는 방법입니다.",
        ]
    ),
]

- 각 InputExample의 첫 번째 항목은 앵커(질문)입니다.
- 두 번째 항목은 포지티브 샘플(관련있는응답)입니다.
- 세 번째 이후 항목들은 하드 네거티브(관련 없지만 구분하기 어려운 응답)입니다.


- 손실함수는 각 쌍에 대해 다른 모든 앵커의 포지티브 샘플들과 모든 하드 네거티브 샘플들을 네거티브로 사용합니다.
- 포지티브와 유사도가 높아지도록 학습되고, 다른 앵커의 포지티브, 하드 네거티브와는 유사도가 낮아지도록 학습됩니다.


- 일반 네거티브(배치 내 무작위 네거티브)
  - 자동으로 배치 내에서 생성됨
  - 대부분 주제가 완전히 다른 무관한 문장들
  - 모델이 구분하기 상대적으로 쉬움
- 하드 네거티브(명시적 네거티브)
  - 사용자가 직접, 의도적으로 선택함
  - 포지티브와 주제는 유사하나 정확한 답변은 아님
  - 미묘한 의미 차이를 포함하여 모델에게 더 큰 도전이 됨
- 예시
  - 질문: 당뇨병의 증상은 무엇인가요?
  - 포지티브: 당뇨병의 주요 증상으로는 갈증 증가, 빈뇨, 체중 감소 등이 있습니다.
  - 하드 네거티브(명시적): 저혈당의 증상으로는 현기증, 발한, 불안감 등이 있습니다. (의료 관련 주제이지만 당뇨병이 아닌 저혈당에 관한 내용)
  - 일반 네거티브(배치 내 자동 선택): 파이썬은 객체지향 프로그래밍 언어입니다.(완전히 다른 주제)


- 하드 네거티브 샘플을 사용하면 모델이 더 미묘한 의미 차이를 학습하게 되어 정확도가 크게 향상될 수 있습니다.
- 가능하다면 일반 네거티브와 하드 네거티브를 병행해 사용하는 것이 이상적입니다.


### 2.3 그 외 학습 성능 향상을 위한 팁

- 학습 데이터와 실전과의 괴리 최소화
  - 학습에 사용할 데이터의 앵커는 실제 RAG에서 사용자가 입력할만한 질문으로 구성해야 합니다. 학습 데이터와 실제 RAG에서 입력될 질문의 차이가 클수록 학습 후의 효용은 떨어지기 마련입니다.
- 데이터 증강
  - 난이도가 높은 하드 네거티브 샘플은 충분히 확보하면 모델이 더 섬세한 의미 차이를 학습할 수 있어 성능 향상에 도움이 됩니다.
- 학습률 조정
  - 학습률을 바꿔가면서 여러 번 학습하여 모델의 성능을 평가하고, 최적의 학습률을 찾아보는 것이 좋습니다.
- 온도 파라미터 조정
  - 손실 함수의 scale파라미터를 조절하면 학습 강도를 세밀하게 조정할 수 있습니다.


## 3. 실전 파인튜닝

### 3.1 데이터 로드하기


In [ ]:
import os
import requests
import json
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from openai import OpenAI
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, losses, InputExample
from sentence_transformers.evaluation import InformationRetrievalEvaluator
import torch
from sklearn.metrics.pairwise import cosine_similarity
import PyPDF2


- os: 환경 변수 설정에 사용되며, 임베딩 모델 파인튜닝 과정에서 오픈AI API키를 설정합니다.
- requests: PDF파일과 같은 학습 데이터를 인터넷에서 다운로드할 때 사용합니다.
- json: API 응답을 처리하거나 구성 설정을 저장/로드할 때 활용합니다.
- pandas: 임베딩 모델 성능 평가 결과를 데이터프레임으로 구성하고 분석하는 데 사용합니다.
- numpy: 벡터 연산을 수행하며 특히 임베딩 벡터 간 코사인 유사도 계산에 사용합니다.
- tqdm: 대용량 데이터셋을 처리할 때 진행 상황을 시각적으로 표시하여 학습 과정을 모니터링 합니다.
- OpenAI: GPT 모델을 사용해 문서로부터 질문을 생성하는 등의 작업에 활용합니다.
- DataLoader: 임베딩 모델 학습 시 배치 단위로 데이터를 효율적으로 로드합니다. 이때, 배치 크기는 성능에 큰 영향을 미칩니다.
- SentenceTransformer: 문장 임베딩 모델의 핵심 라이브러리로, 다양한 사전 학습 모델을 로드하고 파인튜닝합니다.
- losses: MultipleNegativesRankingLoss와 같은 손실함수를 제공하여 임베딩 모델이 관련 문서 쌍은 가깝게, 관련 없는 쌍은 멀게 학습하도록 합니다.
- InputExample: 파인튜닝용 학습데이터 포맷으로, 질문과 관련 문서 쌍을 모델이 이해할 수 있는 형태로 구성합니다.
- InformationRetrievalEvaluator: 파인튜닝된 모델의 검색 성능을 정확도, MRR, NDCG등 다양한 지표로 평가합니다.
- torch: 임베딩 모델의 기본 프레임워크로, 텐서 연산과 GPU 가속을 지원합니다.
- cosine_similarity: 임베딩 벡터 간 유사도를 계산하여 질문에 가장 관련성 높은 문서를 찾는 데 사용합니다.
- PyPDF2: PDF 파일을 읽는데 사용합니다.


In [3]:
from dotenv import load_dotenv

# .env 파일에서 환경 변수 로드
load_dotenv()
# 환경 변수에서 API 키 가져오기
api_key = os.getenv("OPENAI_API_KEY")

### 3.2 하드 네거티브 선정

- 깃허브 저장소에서 일본 ICT 동향 문서와 미국 ICT 동향 문서 두 가지를 다운로드합니다.
- 미국 ICT 동향 문서를 기준으로 임베딩 모델을 학습시키고, 동일한 도메인의 문서인 일본 ICT 동향 문서에 대해 검색 성능을 평가해보겠습니다.
- 실제 현업에서 임베딩 모델을 파인튜닝할 때도 실제 RAG에서 사용할 동일한 도메인의 데이터로 파인튜닝하면 더 좋은 효과를 얻을 수 있습니다.


In [4]:
# PDF 파일 다운로드
urls = [
    "https://raw.githubusercontent.com/langchain-kr/langchain-tutorial/main/Ch09.%20Embedding%20Fine-tuning/ict_japan_2024.pdf",
    "https://raw.githubusercontent.com/langchain-kr/langchain-tutorial/main/Ch09.%20Embedding%20Fine-tuning/ict_usa_2024.pdf",
]

for url in urls:
    filename = url.split("/")[-1]
    response = requests.get(url)
    with open(filename, "wb") as f:
        f.write(response.content)
    print(f"{filename} 다운로드 완료")

ict_japan_2024.pdf 다운로드 완료
ict_usa_2024.pdf 다운로드 완료


In [5]:
def extract_text_from_pdf(pdf_path):
    """PDF 파일에서 텍스트를 추출하는 함수"""
    text_chunks = []
    with open(pdf_path, "rb") as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page_num in range(len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            text = page.extract_text()
            # 페이지 단위로 청크 생성
            if text.strip():
                text = text.strip()
                # 문서 길이가 10자 초과인 경우만 추가
                if len(text) > 10:
                    text_chunks.append(text)
    return text_chunks


# 미국 ICT 동향(학습 데이터)
train_corpus = extract_text_from_pdf("ict_usa_2024.pdf")
print(f"학습 데이터 문서 개수: {len(train_corpus)}")

# 일본 ICT 동향(검증 데이터)
val_corpus = extract_text_from_pdf("ict_japan_2024.pdf")
print(f"검증 데이터 문서 개수: {len(val_corpus)}")

학습 데이터 문서 개수: 26
검증 데이터 문서 개수: 27


- 의미 있는 내용을 보장하기 위해 길이가 10자를 초과하는 텍스트만 청크에 추가합니다.


In [6]:
print("10번 문서:", train_corpus[10])

10번 문서: 13 Ⅰ. ICT 국가 산업 현황
 4.ICT 주요 법령 및 규제
  ② 반도체 과학법 (CHIPS and Science Act)
 반도체 ·전자 기업, $1,660 억 규모 투자 유치 
• 조 바이든 (Joe Biden) 미국 대통령은 2022년 7월 ‘반도체 과학법 (CHIPS and Science Act)’을 승인함  
• 반도체 과학법은 미국의 경쟁력을 강화하고 , 미국의 공급망을 탄력적으로 구축해 국가 안보를 
공고히 하며 국가의 주요 기술에 대한 접근을 지원하는 것을 목표로 함. 법률 제정으로 미국 내 
반도체 생산 제조사 관련 자본 투자는 25%의 세액 공제 혜택이 제공됨
• 미국 백악관은 2023년 8월, 반도체 과학법이 서명된 지 1년 만에 반도체와 전자 관련 기업들이 
1,660 억 달러(221조6,100 억 원)의 투자를 유치했다고 발표함 . 바이든 행정부의 집권 이후 기업들은 
미국 내 반도체와 전자 분야 투자에 총 2,310 억 달러(약 308조 3,850 억 원) 이상의 투자를 약속함
[표 8] 반도체 과학법 주요 이정표 및 진척 현황
주요 이정표 진척 현황 (23년 8월 기준)
미국 반도체 제조 
지원‣ 상무부는 CHIPS 통과 6개월 만에 해당 법에서 제공하는 390억 달러 반도체 제조 
인센티브에 대한 첫 번째 자금 조달 기회 시작
‣ 상무부는 42개 주에서 CHIPS 자금 지원에 관심 있는 460개 기업으로부터 소개서 접수
‣ 상무부는 CHIPS 인센티브 프로그램 관련 140명 이상의 인력으로 구성된 ‘칩스 포 
아메리카 (CHIPS for America)’ 팀 구성
‣ 재무부는 투자에 대한 25% 세액 공제 관련 지침 제공 위해 규칙 제안 발표
국가 안보를 
보호하고 동맹국 및 
파트너와 협력‣ 국무부는 국제 기술 보안 및 혁신 기금 시행 계획 발표
‣ 국방부와 상무부는 CHIPS 투자를 통한 안보 관련 반도체 제조를 위해 협력 확대 협의
‣ CHIPS 를 시행하면서 상무부는 여러 파트너 및 동맹국과 긴밀한 접촉